# Expectation-Maximization for ambiguous transcript assignment

This is a **synthetic teaching example**, not biological evidence and not a reimplementation of Salmon. It isolates the core idea: fragments from a shared exon can be compatible with more than one transcript, so their assignments and transcript abundances must be estimated together. Salmon additionally models effective length, fragment properties, sequence/GC bias, rich equivalence classes, and other details.

## Toy model and likelihood

There are two transcripts. Class 1 can arise only from transcript 1, class 2 is a shared-exon class, and class 3 can arise only from transcript 2. We observe 30, 60, and 10 fragments in those classes. The emission matrix below gives $P(class\mid transcript)$; each column sums to one.

For transcript proportions $\theta_1+\theta_2=1$, class probability is $p_c=\sum_t a_{ct}\theta_t$, and the multinomial log-likelihood (apart from a constant) is $\ell(\theta)=\sum_c n_c\log(p_c)$.

**E-step:** allocate each class fractionally with responsibility $r_{ct}=a_{ct}\theta_t / \sum_j a_{cj}\theta_j$. **M-step:** update $\theta_t=\sum_c n_c r_{ct}/\sum_c n_c$. Iterate until the largest abundance change is tiny.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from fastq_pipeline.em_toy import run_em

class_names = ['unique to transcript 1', 'shared exon', 'unique to transcript 2']
transcript_names = ['transcript 1', 'transcript 2']
counts = np.array([30, 60, 10])
emission = np.array([[0.5, 0.0], [0.5, 0.5], [0.0, 0.5]])
pd.DataFrame(emission, index=class_names, columns=transcript_names).assign(observed_fragments=counts)

In [ ]:
history, final_allocations = run_em(counts, emission, np.array([0.5, 0.5]))
history.head(8)

In [ ]:
final = history.iloc[-1]
allocation_table = pd.DataFrame(final_allocations, index=class_names, columns=transcript_names)
print(f"Converged after {int(final['iteration'])} iterations")
print(f"Final proportions: transcript 1={final['theta_1']:.4f}, transcript 2={final['theta_2']:.4f}")
allocation_table

In [ ]:
output_dir = ROOT / 'notebooks' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.iteration, history.theta_1, marker='o', ms=3, label='transcript 1')
axes[0].plot(history.iteration, history.theta_2, marker='o', ms=3, label='transcript 2')
axes[0].set(xlabel='EM iteration', ylabel='Estimated transcript proportion', title='Abundance convergence')
axes[0].legend(frameon=False)
axes[1].plot(history.iteration, history.log_likelihood, marker='o', ms=3, color='#7b3294')
axes[1].set(xlabel='EM iteration', ylabel='Log-likelihood', title='Likelihood does not decrease')
fig.suptitle('Synthetic two-transcript EM example')
fig.tight_layout()
fig.savefig(output_dir / 'em_toy_convergence.png', dpi=180)
plt.show()

## Interpretation

The unique classes anchor the mixture. Starting from equal abundance, the shared fragments are first split equally; after each M-step favors transcript 1, the next E-step assigns a larger fraction of shared fragments to transcript 1. The estimates converge when another E/M cycle produces no meaningful change. This demonstrates the feedback at the heart of transcript quantification, but the numerical estimates should not be compared directly with Salmon output because the toy emission model omits the richer assumptions listed above.